# CH04 — Recursión

Material del curso basado en Goodrich, Tamassia & Goldwasser. El código de ejemplo está en `goodrich/ch04`.

## Resumen del capítulo 4 — Recursión (Goodrich, Tamassia & Goldwasser)

### 4.1 ¿Qué es la recursión?

Una función es **recursiva** cuando se llama a sí misma para resolver instancias más pequeñas del mismo problema. Toda función recursiva bien definida necesita:

- **Caso base:** una o más instancias del problema que se resuelven directamente, sin más llamadas recursivas. Sin caso base, la recursión nunca termina.
- **Caso recursivo:** expresa el problema en términos de una o más instancias más pequeñas del mismo problema, que eventualmente llegan al caso base.

Ejemplo mínimo — `factorial`:

```python
def factorial(n):
    if n == 0:        # caso base
        return 1
    else:              # caso recursivo
        return n * factorial(n-1)
```

### 4.2 La pila de llamadas (activation stack)

Cada llamada a una función recursiva genera un nuevo **marco de activación** (*activation record*), con sus propios parámetros y variables locales. Python mantiene una pila de estos marcos; cuando una llamada retorna, su marco se descarta y el control vuelve al marco que la invocó.

Esto explica por qué la recursión tiene un costo de memoria proporcional a la **profundidad** de la recursión (número de llamadas anidadas), y por qué existe un límite práctico: Python impone por defecto un límite de ~1000 llamadas anidadas (`sys.getrecursionlimit()`), lanzando `RecursionError` si se excede.

### 4.3 Recursión lineal

Cada llamada genera **como máximo una** llamada recursiva adicional. La cadena de llamadas es una lista, no un árbol.

| Ejemplo | Archivo | Idea |
|---|---|---|
| Suma de los primeros *n* elementos | `linear_sum.py` | `linear_sum(S, n) = linear_sum(S, n-1) + S[n-1]` |
| Invertir una secuencia | `reverse.py` | intercambia extremos y recurre sobre el resto |
| Potencia rápida | `power_fast.py` | *divide y vencerás*: calcula `x**(n//2)` una sola vez y lo eleva al cuadrado |
| Potencia lenta | `power_slow.py` | `x * power(x, n-1)`, O(n) llamadas |
| Búsqueda binaria | `binary_search.py` | recurre solo sobre la mitad relevante del arreglo |

**Complejidad:** `linear_sum`, `reverse` y `power_slow` son O(n) (una llamada por elemento). `power_fast` reduce el exponente a la mitad en cada llamada, logrando O(log n). `binary_search` también es O(log n) por la misma razón: cada llamada descarta la mitad de las opciones restantes.

### 4.4 Buena vs. mala recursión: Fibonacci

`fibonacci.py` contrasta dos enfoques:

- `bad_fibonacci(n)`: usa **recursión binaria** ingenua (`bad_fibonacci(n-1) + bad_fibonacci(n-2)`), recalculando los mismos subproblemas una y otra vez → **O(2ⁿ)**, exponencial.
- `good_fibonacci(n)`: usa recursión **lineal**, calculando el par `(F(n), F(n-1))` en una sola llamada recursiva por nivel → **O(n)**.

Esta comparación ilustra que el número de llamadas recursivas por caso (binaria vs. lineal) puede cambiar la complejidad de exponencial a lineal.

### 4.5 Recursión binaria

Cada llamada puede generar **dos** llamadas recursivas. Es la base de algoritmos de tipo *divide y vencerás*.

- `binary_sum.py`: divide el arreglo en dos mitades, suma cada mitad recursivamente y combina los resultados. Complejidad O(n), pero con profundidad de recursión O(log n) — mucho menor que la versión lineal, lo cual reduce el uso de pila.

### 4.6 Recursión múltiple

Una llamada puede generar **más de dos** llamadas recursivas (o un número variable, dependiente de los datos).

- `disk_usage.py`: recorre un árbol de directorios; cada carpeta puede generar una llamada recursiva por cada archivo/subcarpeta que contiene.
- `ruler.py` (`draw_interval`): genera una regla inglesa (*English ruler*) mediante dos llamadas recursivas simétricas por nivel, ejemplo clásico de patrón fractal recursivo con efectos de impresión en lugar de retorno de valores.
- `unique_bad.py`: ejemplo de **mala práctica** — comprueba si hay duplicados dividiendo el problema en dos llamadas recursivas que se solapan, resultando en complejidad exponencial. Sirve como advertencia: no toda descomposición recursiva es eficiente.

### 4.7 Recursión vs. iteración

Toda recursión se puede reescribir de forma iterativa (y viceversa). El capítulo compara directamente:

- `reverse` (recursiva) vs. `reverse_iterative.py`
- `binary_search` (recursiva) vs. `binary_search_iterative.py`

Motivos para preferir la versión iterativa:

- Evita el límite de profundidad de recursión de Python.
- Evita el overhead de crear/destruir marcos de activación.
- En **recursión de cola** (*tail recursion*, cuando la llamada recursiva es la última operación, como en `binary_search`), la conversión a un bucle `while` es directa y elimina por completo el uso extra de pila.

### 4.8 Ideas clave para recordar

1. Toda función recursiva necesita al menos un caso base alcanzable.
2. El número de llamadas recursivas por invocación (una, dos, o varias) determina si hablamos de recursión **lineal**, **binaria** o **múltiple**, y afecta directamente la complejidad temporal y espacial.
3. Recursiones que **recalculan** los mismos subproblemas (como `bad_fibonacci` o `unique_bad`) pueden ser exponencialmente más lentas que una alternativa que evita ese solapamiento.
4. La profundidad de recursión consume memoria de pila; Python limita esa profundidad por defecto.
5. Cualquier algoritmo recursivo puede reescribirse de forma iterativa; la recursión de cola es el caso más sencillo de convertir.

## Conceptos teóricos: verificación con código

A continuación se importan y ejecutan los ejemplos de `goodrich/ch04` para verificar el comportamiento descrito arriba.

In [ ]:
from goodrich.ch04.factorial import factorial
from goodrich.ch04.fibonacci import bad_fibonacci, good_fibonacci
from goodrich.ch04.linear_sum import linear_sum
from goodrich.ch04.binary_search import binary_search
from goodrich.ch04.binary_sum import binary_sum
from goodrich.ch04.reverse import reverse

In [ ]:
factorial(5), linear_sum([4, 3, 6, 2, 8], 5), binary_search([2,4,5,7,8,9,12,14,17,19,22,25,27,28,33,37], 22, 0, 15)

### Comparación de tiempos: `bad_fibonacci` vs `good_fibonacci`

`bad_fibonacci` recalcula los mismos subproblemas exponencialmente muchas veces; `good_fibonacci` los calcula una sola vez por nivel de recursión.

In [ ]:
from time import time

n = 28
a = time()
r1 = bad_fibonacci(n)
b = time()
print(f"bad_fibonacci({n})  = {r1:<10}  {b-a:.4f} s")

a = time()
r2, _ = good_fibonacci(n)
b = time()
print(f"good_fibonacci({n}) = {r2:<10}  {b-a:.4f} s")

### Recursión de cola: profundidad de `sys.getrecursionlimit()`

Ejemplo de cómo `linear_sum` (recursión lineal, no de cola) puede agotar la pila para `n` grande, mientras que su versión iterativa no tiene ese problema.

In [ ]:
import sys
print("Límite de recursión de Python:", sys.getrecursionlimit())

try:
    linear_sum(list(range(5000)), 5000)
except RecursionError as e:
    print("RecursionError:", e)

# Ejercicios en clase

## Vamos a ordenar una lista de manera recursiva

In [21]:
def ordenar_lista(L):
    if len(L) == 1:
        return L

    L1 = ordenar_lista(L[:-1])
    y = L[-1]
    posicion = 0
    for i in range(len(L) -1):
        if L[i] < y:
            posicion += 1
    return L1[:posicion] + [y] + L1[posicion:]

### Un test de tiempo: Nuestro algoritmo vs sort

In [11]:
# Vamos a crear la lista [1, -1, 2, -2 , ..., 500000, -50000] para hacer un test de tiempo
L1 = list(range(1,500001))
L2 = []
for i in L1:
    L2.append(i)
    L2.append(-i)

L2[:10]

[1, -1, 2, -2, 3, -3, 4, -4, 5, -5]

In [17]:
len(L2)

1000000

In [ ]:
# Demora menos de un segundo en ejecutar.
L2.sort()

In [ ]:
# No ejecutar o va romper su pc
ordenar_lista(L2)

## Vamos a realizar el ejercicio 4.19 y el `suma_isabel`

In [2]:
# 4.19
def pares_izquierda(L):
    if len(L) == 1:
        return L
    L1 = pares_izquierda(L[:-1])
    if L[-1]%2 == 0:
        return [L[-1]] + L1
    else:
        return L1 + [L[-1]]

In [4]:
L = [1,2,3,4,5,6]
pares_izquierda(L)

[6, 4, 2, 1, 3, 5]

In [5]:
L

[1, 2, 3, 4, 5, 6]

In [12]:
def suma_isabel(L):
    if len(L) == 1:
        return L[0]

    lista = []
    for i in range(0, len(L), 2):
        lista.append(L[i] + L[i+1])

    return suma_isabel(lista)

In [13]:
suma_isabel([1,2,3,4,5,6,7,8])

36